# EnzAIme neural activity feasibility baseline
This notebook does not train thermostability or establish mutation improvements. No paired images are included. Read docs/multimodal_training.md. Enable a T4 GPU and Internet. Upload neural_kaggle_starter.zip as a private Kaggle dataset before running.


In [ ]:
from pathlib import Path
import os, shutil, zipfile

inputs = Path('/kaggle/input')
root = Path('/kaggle/working/EnzAIme_ML')
required = ['backend/requirements-neural.txt',
            'scripts/prepare_neural_features.py',
            'scripts/train_neural_oracle.py',
            'data/processed/tpa_pet_ml_confirmed.csv']

# Kaggle may unpack an uploaded ZIP into the dataset directory.
sources = sorted({p.parent.parent for p in inputs.rglob('requirements-neural.txt')
                  if all((p.parent.parent / name).is_file() for name in required)})
archives = sorted(inputs.rglob('neural_kaggle_starter.zip'))
if len(sources) == 1:
    root.mkdir(parents=True, exist_ok=True)
    for name in ['backend', 'scripts', 'data', 'docs', 'notebooks']:
        source = sources[0] / name
        if source.is_dir():
            shutil.copytree(source, root / name, dirs_exist_ok=True)
    print('Copied extracted starter from:', sources[0])
elif not sources and len(archives) == 1:
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        if not all((root / name).resolve().is_relative_to(root.resolve())
                   for name in archive.namelist()):
            raise ValueError('ZIP contains a path outside the working directory')
        archive.extractall(root)
    print('Extracted:', archives[0])
else:
    print('Input entries:', [str(p) for p in inputs.iterdir()] if inputs.exists() else [])
    print('Matching extracted starters:', [str(p) for p in sources])
    print('Matching ZIPs:', [str(p) for p in archives])
    raise RuntimeError('Attach one EnzAIme starter dataset using Add Input; remove duplicate starter datasets.')

missing = [name for name in required if not (root / name).is_file()]
if missing:
    raise RuntimeError(f'Starter is incomplete: {missing}')
os.chdir(root)
print('Ready:', Path.cwd())


In [ ]:
%pip install -r backend/requirements-neural.txt


In [ ]:
import torch
print('GPU count:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Enable GPU in Kaggle settings'
!python -m scripts.prepare_neural_features


## Train and evaluate
Exact sequence groups are held apart. Homologous proteins and publications may still cross splits. This is a feasibility check, not a final generalization result.


In [ ]:
!python -m scripts.train_neural_oracle --sequence-only --output models/neural
import json
print(json.dumps(json.loads(Path('models/neural/manifest.json').read_text()), indent=2))


## Optional second T4
Use separate seeds/outputs on the two GPUs. Do not repeatedly choose models based on held-out test scores. Image fusion requires a verified view manifest; see the guide.


In [ ]:
import zipfile
output = Path('/kaggle/working/enzaime_neural_results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['models/neural', 'data/neural_features']:
        for file in Path(folder).rglob('*'):
            if file.is_file(): archive.write(file, str(file))
print('Download and return:', output)
